<a href="https://colab.research.google.com/github/sabrian-lab/Metode-formal/blob/main/Formal_Method_Case_Study.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pemodelan Sistem Secara Formal
## Studi Kasus: Sistem Rekomendasi Perkuliahan

Mata kuliah ini membahas bagaimana membangun model sistem secara formal dan menerjemahkannya ke dalam simulasi Python di Google Colab.

### Tujuan pembelajaran
Setelah mengikuti notebook ini, mahasiswa diharapkan mampu:
1. mengidentifikasi domain, relasi, fungsi, dan constraint dalam sistem formal,
2. menuliskan spesifikasi sederhana dengan gaya Z formal specification,
3. memeriksa invariant atau constraint sistem,
4. mensimulasikan model formal menggunakan Python,
5. menganalisis validitas rekomendasi yang dihasilkan sistem.

# Pengantar

Google Colab tidak digunakan untuk mengeksekusi bahasa Z secara langsung.  
Namun, Google Colab sangat cocok untuk:

- menuliskan spesifikasi formal dalam bentuk teks,
- memodelkan domain dan relasi,
- memeriksa invariant atau constraint,
- mensimulasikan perilaku sistem dengan Python.

Dengan pendekatan ini, mahasiswa tidak hanya membaca simbol formal, tetapi juga memahami makna operasionalnya.

# Kasus: Sistem Rekomendasi Perkuliahan

Sistem Rekomendasi Perkuliahan adalah sistem yang memberikan rekomendasi mata kuliah kepada mahasiswa berdasarkan:

- data akademik mahasiswa,
- aturan kurikulum,
- prasyarat mata kuliah,
- kemampuan mahasiswa,
- preferensi mahasiswa,
- beban studi,
- potensi keberhasilan mahasiswa.

### Komponen utama
- Mahasiswa
- Mata kuliah
- Struktur kurikulum
- Model evaluasi kemampuan
- Fungsi rekomendasi

### Constraint utama
1. Mata kuliah yang direkomendasikan harus memenuhi seluruh prasyarat.
2. Sistem hanya merekomendasikan mata kuliah yang layak secara akademik.
3. Sistem tidak merekomendasikan mata kuliah yang telah diselesaikan.
4. Rekomendasi harus mempertimbangkan kemampuan mahasiswa dan tingkat kesulitan mata kuliah.

# Spesifikasi Formal Gaya Z

## Domain Dasar

```text
[STUDENT, COURSE]


---

## Cell 5 — Z Formal Specification: State dan Fungsi
**Tipe:** Markdown

```markdown
# State dan Fungsi Formal

```text
passed: STUDENT \fun \power COURSE
prerequisite: COURSE \fun \power COURSE
ability: STUDENT \fun \mathbb{R}
difficulty: COURSE \fun \mathbb{R}
workload: COURSE \fun \mathbb{N}
preference: STUDENT \cross COURSE \fun \mathbb{R}

In [ ]:
from typing import Dict, Set, List, Tuple

In [ ]:
Student = str
Course = str
Score = float

In [ ]:
students: Set[Student] = {"Ali", "Budi", "Citra"}

courses: Set[Course] = {
    "Matematika Diskrit",
    "Struktur Data",
    "Algoritma",
    "Basis Data",
    "Machine Learning"
}

prerequisite: Dict[Course, Set[Course]] = {
    "Matematika Diskrit": set(),
    "Struktur Data": {"Matematika Diskrit"},
    "Algoritma": {"Struktur Data"},
    "Basis Data": {"Matematika Diskrit"},
    "Machine Learning": {"Algoritma", "Basis Data"}
}

passed: Dict[Student, Set[Course]] = {
    "Ali": {"Matematika Diskrit", "Struktur Data"},
    "Budi": {"Matematika Diskrit"},
    "Citra": {"Matematika Diskrit", "Struktur Data", "Basis Data", "Algoritma"}
}

ability: Dict[Student, float] = {
    "Ali": 80.0,
    "Budi": 68.0,
    "Citra": 90.0
}

difficulty: Dict[Course, float] = {
    "Matematika Diskrit": 60.0,
    "Struktur Data": 72.0,
    "Algoritma": 78.0,
    "Basis Data": 75.0,
    "Machine Learning": 88.0
}

workload: Dict[Course, int] = {
    "Matematika Diskrit": 3,
    "Struktur Data": 3,
    "Algoritma": 3,
    "Basis Data": 3,
    "Machine Learning": 4
}

preference: Dict[Tuple[Student, Course], float] = {
    ("Ali", "Algoritma"): 0.85,
    ("Ali", "Basis Data"): 0.75,
    ("Ali", "Machine Learning"): 0.95,
    ("Budi", "Struktur Data"): 0.70,
    ("Budi", "Basis Data"): 0.80,
    ("Budi", "Algoritma"): 0.65,
    ("Citra", "Machine Learning"): 0.90,
    ("Citra", "Basis Data"): 0.70
}

In [ ]:
print("Students:", students)
print("Courses:", courses)
print("Prerequisite:", prerequisite)
print("Passed:", passed)
print("Ability:", ability)
print("Difficulty:", difficulty)
print("Workload:", workload)

Students: {'Budi', 'Citra', 'Ali'}
Courses: {'Struktur Data', 'Basis Data', 'Algoritma', 'Machine Learning', 'Matematika Diskrit'}
Prerequisite: {'Matematika Diskrit': set(), 'Struktur Data': {'Matematika Diskrit'}, 'Algoritma': {'Struktur Data'}, 'Basis Data': {'Matematika Diskrit'}, 'Machine Learning': {'Algoritma', 'Basis Data'}}
Passed: {'Ali': {'Struktur Data', 'Matematika Diskrit'}, 'Budi': {'Matematika Diskrit'}, 'Citra': {'Struktur Data', 'Basis Data', 'Algoritma', 'Matematika Diskrit'}}
Ability: {'Ali': 80.0, 'Budi': 68.0, 'Citra': 90.0}
Difficulty: {'Matematika Diskrit': 60.0, 'Struktur Data': 72.0, 'Algoritma': 78.0, 'Basis Data': 75.0, 'Machine Learning': 88.0}
Workload: {'Matematika Diskrit': 3, 'Struktur Data': 3, 'Algoritma': 3, 'Basis Data': 3, 'Machine Learning': 4}


In [ ]:
def check_domain_consistency():
    assert set(prerequisite.keys()).issubset(courses), "Ada key prerequisite di luar domain course"
    assert set(difficulty.keys()).issubset(courses), "Ada key difficulty di luar domain course"
    assert set(workload.keys()).issubset(courses), "Ada key workload di luar domain course"
    assert set(passed.keys()).issubset(students), "Ada key passed di luar domain student"
    assert set(ability.keys()).issubset(students), "Ada key ability di luar domain student"

    for s, passed_courses in passed.items():
        assert passed_courses.issubset(courses), f"Ada passed course milik {s} yang tidak valid"

    for c, prereq_courses in prerequisite.items():
        assert prereq_courses.issubset(courses), f"Ada prerequisite course untuk {c} yang tidak valid"

    for (s, c), pref in preference.items():
        assert s in students, f"Student {s} pada preference tidak valid"
        assert c in courses, f"Course {c} pada preference tidak valid"
        assert 0.0 <= pref <= 1.0, f"Preference {s,c} harus di antara 0 dan 1"

    print("Semua domain konsisten.")

In [ ]:
check_domain_consistency()

Semua domain konsisten.


# Pemeriksaan Constraint Formal

Kita implementasikan constraint utama:
1. prasyarat harus terpenuhi,
2. mata kuliah belum pernah lulus,
3. kemampuan mahasiswa harus memadai.

In [ ]:
def has_completed_prerequisites(student: Student, course: Course) -> bool:
    return prerequisite[course].issubset(passed[student])

def not_yet_completed(student: Student, course: Course) -> bool:
    return course not in passed[student]

def ability_matches_difficulty(student: Student, course: Course) -> bool:
    return ability[student] >= difficulty[course]

def is_academically_eligible(student: Student, course: Course) -> bool:
    return (
        has_completed_prerequisites(student, course)
        and not_yet_completed(student, course)
        and ability_matches_difficulty(student, course)
    )

In [ ]:
for s in students:
    print(f"\nMahasiswa: {s}")
    for c in courses:
        print(
            f"{c:20s} | "
            f"prasyarat={has_completed_prerequisites(s,c)} | "
            f"belum_lulus={not_yet_completed(s,c)} | "
            f"ability_ok={ability_matches_difficulty(s,c)} | "
            f"eligible={is_academically_eligible(s,c)}"
        )


Mahasiswa: Budi
Struktur Data        | prasyarat=True | belum_lulus=True | ability_ok=False | eligible=False
Basis Data           | prasyarat=True | belum_lulus=True | ability_ok=False | eligible=False
Algoritma            | prasyarat=False | belum_lulus=True | ability_ok=False | eligible=False
Machine Learning     | prasyarat=False | belum_lulus=True | ability_ok=False | eligible=False
Matematika Diskrit   | prasyarat=True | belum_lulus=False | ability_ok=True | eligible=False

Mahasiswa: Citra
Struktur Data        | prasyarat=True | belum_lulus=False | ability_ok=True | eligible=False
Basis Data           | prasyarat=True | belum_lulus=False | ability_ok=True | eligible=False
Algoritma            | prasyarat=True | belum_lulus=False | ability_ok=True | eligible=False
Machine Learning     | prasyarat=True | belum_lulus=True | ability_ok=True | eligible=True
Matematika Diskrit   | prasyarat=True | belum_lulus=False | ability_ok=True | eligible=False

Mahasiswa: Ali
Struktur Data      

# Fungsi Evaluasi

Materi  menyebutkan bahwa score dapat mempertimbangkan:
- probabilitas keberhasilan,
- preferensi,
- beban studi.

Di sini kita buat model sederhana agar mahasiswa mudah memahami konsepnya.

In [ ]:
def success_probability(student: Student, course: Course) -> float:
    gap = ability[student] - difficulty[course]

    if gap >= 10:
        return 0.95
    elif gap >= 0:
        return 0.80
    elif gap >= -5:
        return 0.60
    else:
        return 0.30

def get_preference(student: Student, course: Course) -> float:
    return preference.get((student, course), 0.50)

def compute_score(student: Student, course: Course) -> float:
    sp = success_probability(student, course)
    pref = get_preference(student, course)
    wl = workload[course]

    # model sederhana
    return (0.5 * sp) + (0.4 * pref) - (0.1 * (wl / 4))

In [ ]:
for s in students:
    print(f"\nMahasiswa: {s}")
    for c in courses:
        sc = compute_score(s, c)
        print(f"{c:20s} -> {sc:.3f}")


Mahasiswa: Budi
Struktur Data        -> 0.505
Basis Data           -> 0.395
Algoritma            -> 0.335
Machine Learning     -> 0.250
Matematika Diskrit   -> 0.525

Mahasiswa: Citra
Struktur Data        -> 0.600
Basis Data           -> 0.680
Algoritma            -> 0.600
Machine Learning     -> 0.660
Matematika Diskrit   -> 0.600

Mahasiswa: Ali
Struktur Data        -> 0.525
Basis Data           -> 0.625
Algoritma            -> 0.665
Machine Learning     -> 0.430
Matematika Diskrit   -> 0.600


# Fungsi Rekomendasi

Mata kuliah hanya akan direkomendasikan jika:
- lolos seluruh constraint,
- lalu dinilai menggunakan fungsi score.

In [ ]:
def recommend_courses(student: Student) -> List[Tuple[Course, Score]]:
    recommendations = []

    for course in courses:
        if is_academically_eligible(student, course):
            score = compute_score(student, course)
            recommendations.append((course, score))

    recommendations.sort(key=lambda x: x[1], reverse=True)
    return recommendations

In [ ]:
for s in students:
    print(f"\n=== Rekomendasi untuk {s} ===")
    recs = recommend_courses(s)

    if not recs:
        print("Tidak ada mata kuliah yang layak direkomendasikan.")
    else:
        for course, score in recs:
            print(f"{course:20s} | score = {score:.3f}")


=== Rekomendasi untuk Budi ===
Tidak ada mata kuliah yang layak direkomendasikan.

=== Rekomendasi untuk Citra ===
Machine Learning     | score = 0.660

=== Rekomendasi untuk Ali ===
Algoritma            | score = 0.665
Basis Data           | score = 0.625


# Analisis Keputusan Sistem

Agar mahasiswa memahami semantik formal, kita buat fungsi penjelas keputusan sistem.

In [ ]:
def explain_course_decision(student: Student, course: Course):
    print(f"Mahasiswa : {student}")
    print(f"Mata kuliah: {course}")
    print("-" * 40)

    prereq_ok = has_completed_prerequisites(student, course)
    not_done = not_yet_completed(student, course)
    ability_ok = ability_matches_difficulty(student, course)

    print(f"Prasyarat terpenuhi     : {prereq_ok}")
    print(f"Belum pernah diselesaikan: {not_done}")
    print(f"Kemampuan memadai       : {ability_ok}")

    if is_academically_eligible(student, course):
        print(f"Status                  : DIREKOMENDASIKAN")
        print(f"Success probability     : {success_probability(student, course):.2f}")
        print(f"Preference              : {get_preference(student, course):.2f}")
        print(f"Workload                : {workload[course]}")
        print(f"Final score             : {compute_score(student, course):.3f}")
    else:
        print("Status                  : TIDAK DIREKOMENDASIKAN")

In [ ]:
explain_course_decision("Ali", "Machine Learning")
print()
explain_course_decision("Budi", "Algoritma")
print()
explain_course_decision("Citra", "Machine Learning")

Mahasiswa : Ali
Mata kuliah: Machine Learning
----------------------------------------
Prasyarat terpenuhi     : False
Belum pernah diselesaikan: True
Kemampuan memadai       : False
Status                  : TIDAK DIREKOMENDASIKAN

Mahasiswa : Budi
Mata kuliah: Algoritma
----------------------------------------
Prasyarat terpenuhi     : False
Belum pernah diselesaikan: True
Kemampuan memadai       : False
Status                  : TIDAK DIREKOMENDASIKAN

Mahasiswa : Citra
Mata kuliah: Machine Learning
----------------------------------------
Prasyarat terpenuhi     : True
Belum pernah diselesaikan: True
Kemampuan memadai       : True
Status                  : DIREKOMENDASIKAN
Success probability     : 0.80
Preference              : 0.90
Workload                : 4
Final score             : 0.660


# Demonstrasi Pelanggaran Constraint

Bagian ini berguna untuk menunjukkan bahwa formal model dapat mendeteksi kondisi yang tidak valid.

In [ ]:
# Contoh: mahasiswa baru yang belum memenuhi prasyarat tetapi dipaksa mengambil Machine Learning
student_test = "Budi"
course_test = "Machine Learning"

print("Prasyarat terpenuhi?", has_completed_prerequisites(student_test, course_test))
print("Belum pernah lulus?", not_yet_completed(student_test, course_test))
print("Kemampuan memadai?", ability_matches_difficulty(student_test, course_test))
print("Eligible?", is_academically_eligible(student_test, course_test))

Prasyarat terpenuhi? False
Belum pernah lulus? True
Kemampuan memadai? False
Eligible? False


# Invariant Checker

Kita dapat memandang validitas rekomendasi sebagai invariant sistem:
setiap elemen hasil rekomendasi harus memenuhi constraint formal.

In [ ]:
def check_recommendation_invariant(student: Student, recommendations: List[Tuple[Course, Score]]):
    recommended_courses = [course for course, _ in recommendations]

    # non redundansi
    assert len(recommended_courses) == len(set(recommended_courses)), "Ada duplikasi rekomendasi"

    for course in recommended_courses:
        assert has_completed_prerequisites(student, course), f"Prasyarat tidak terpenuhi untuk {course}"
        assert not_yet_completed(student, course), f"{course} sudah pernah diselesaikan"
        assert ability_matches_difficulty(student, course), f"Kemampuan tidak memadai untuk {course}"

    print(f"Invariant rekomendasi untuk {student} terpenuhi.")

In [ ]:
for s in students:
    recs = recommend_courses(s)
    check_recommendation_invariant(s, recs)

Invariant rekomendasi untuk Budi terpenuhi.
Invariant rekomendasi untuk Citra terpenuhi.
Invariant rekomendasi untuk Ali terpenuhi.


# Latihan Mahasiswa

Modifikasi sistem agar mempertimbangkan batas maksimum beban studi semester, misalnya 6 SKS atau 9 SKS.

Tambahkan aturan bahwa mata kuliah dengan workload tinggi hanya boleh direkomendasikan jika ability mahasiswa lebih tinggi dari threshold tertentu.

Modifikasi fungsi score agar bobot preference lebih besar daripada success probability.

Tambahkan field baru, misalnya:
- minat topik,
- semester mahasiswa,
- kategori mata kuliah wajib atau pilihan.

# Penutup

Pemodelan formal membantu kita mendefinisikan sistem secara presisi.
Python di Google Colab membantu kita melihat makna operasional dari model tersebut.

Dengan demikian, mahasiswa tidak hanya memahami simbol formal, tetapi juga memahami bagaimana constraint menjaga validitas sistem.
